# 🚀 Multi-Model Fallback Router — Fine-Tuning

In [5]:
# =====================================================
# STEP 1: Install required libraries
# transformers -> Load Llama model
# peft -> Apply LoRA fine tuning
# bitsandbytes -> Optimize GPU memory
# trl -> Train language models
# =====================================================

In [3]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.0 MB/s eta 0:00:00


In [6]:
# =====================================================
# STEP 2: Import required libraries and check GPU
#
# torch -> GPU and deep learning operations
# pandas -> Dataset handling
# transformers -> Load Llama 3.2 model
# peft -> LoRA fine tuning
# datasets -> Convert data for training
# =====================================================

In [7]:
import torch
import pandas as pd

from datasets import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print("✅ Libraries imported successfully")

print("--------------------------------")

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

✅ Libraries imported successfully
--------------------------------
CUDA Available: True
GPU Name: Tesla T4


In [9]:
# =====================================================
# STEP 3: Connect Google Drive
# =====================================================

from google.colab import drive

drive.mount('/content/drive')

print("✅ Google Drive connected successfully")

Mounted at /content/drive
✅ Google Drive connected successfully


In [10]:
# =====================================================
# STEP 4: Create Project Directory Structure
# =====================================================

import os

# Main project path inside Google Drive
PROJECT_PATH = "/content/drive/MyDrive/llama3.2-router-finetuning"


# Create folders
folders = [
    "dataset",
    "checkpoints",
    "final_model"
]


for folder in folders:
    folder_path = f"{PROJECT_PATH}/{folder}"
    os.makedirs(folder_path, exist_ok=True)


print("✅ Project folders created successfully")

print("Project Location:")
print(PROJECT_PATH)

✅ Project folders created successfully
Project Location:
/content/drive/MyDrive/llama3.2-router-finetuning


In [11]:
# =====================================================
# STEP 5: Upload Router Training Dataset
# =====================================================

from google.colab import files

uploaded = files.upload()

print("✅ Dataset uploaded successfully")

Saving training_dataset_69.csv to training_dataset_69.csv
✅ Dataset uploaded successfully


In [12]:
# =====================================================
# STEP 6: Load and Inspect Dataset
# =====================================================

import pandas as pd


# Replace name if your CSV has different name
df = pd.read_csv(list(uploaded.keys())[0])


print("Dataset Shape:")
print(df.shape)


print("\nColumns:")
print(df.columns)


print("\nSample Data:")
df.head()

Dataset Shape:
(2000, 2)

Columns:
Index(['prompt', 'category'], dtype='object')

Sample Data:


,prompt,category
0,Solve the system of equations: 69x + 151y = {n...,math
1,Recommend some travel destinations for someone...,casual_chat
2,Tell me a story about a programmer who discove...,casual_chat
3,Explain how the concept of microservices diffe...,research
4,Simplify the algebraic expression: (x^2 - 9) /...,math


In [13]:
# =====================================================
# STEP 7: Convert Dataset into Llama 3.2 Router Format
# =====================================================


import json


def format_router_dataset(row):

    # Prompt given to Llama during training
    instruction = f"""
### Instruction:
You are an intelligent AI model router.

Analyze the user's request and decide:
1. Task category
2. Best model to handle the request
3. Backup model if the first model fails

Return only JSON output.

### User Request:
{row['prompt']}

### Response:
"""


    # Expected output that Llama learns
    response = {
        "task_type": row["category"],

        "primary_model": (
            "llama3.2:1b"
            if row["category"] in [
                "casual_chat",
                "math",
                "summarization"
            ]
            else "mixtral"
        ),

        "fallback_model": "qwen"
    }


    return instruction + json.dumps(response)



# Apply formatting
df["text"] = df.apply(
    format_router_dataset,
    axis=1
)


print("✅ Dataset converted into Llama format")

print("------------------------------------")

print("Total training examples:", len(df))

print("------------------------------------")

print(df["text"].iloc[0])

✅ Dataset converted into Llama format
------------------------------------
Total training examples: 2000
------------------------------------

### Instruction:
You are an intelligent AI model router.

Analyze the user's request and decide:
1. Task category
2. Best model to handle the request
3. Backup model if the first model fails

Return only JSON output.

### User Request:
Solve the system of equations: 69x + 151y = {num} and {num}x - {num}y = {num}.

### Response:
{"task_type": "math", "primary_model": "llama3.2:1b", "fallback_model": "qwen"}


In [14]:
# =====================================================
# STEP 8: Convert Pandas DataFrame to HuggingFace Dataset
# =====================================================


from datasets import Dataset


# Keep only training text column
training_df = df[["text"]]


# Convert pandas dataframe to HF Dataset
train_dataset = Dataset.from_pandas(training_df)


print("✅ Converted to HuggingFace Dataset")

print("--------------------------------")

print(train_dataset)


print("--------------------------------")

print("Sample:")

print(train_dataset[0]["text"])

✅ Converted to HuggingFace Dataset
--------------------------------
Dataset({
    features: ['text'],
    num_rows: 2000
})
--------------------------------
Sample:

### Instruction:
You are an intelligent AI model router.

Analyze the user's request and decide:
1. Task category
2. Best model to handle the request
3. Backup model if the first model fails

Return only JSON output.

### User Request:
Solve the system of equations: 69x + 151y = {num} and {num}x - {num}y = {num}.

### Response:
{"task_type": "math", "primary_model": "llama3.2:1b", "fallback_model": "qwen"}


In [21]:
# =====================================================
# STEP 9: Load Llama 3.2 1B Model
# =====================================================


import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


# Base model from Hugging Face
model_name = "meta-llama/Llama-3.2-1B"


# 4-bit configuration
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)



# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


# Add padding token
tokenizer.pad_token = tokenizer.eos_token



# Load Llama model
model = AutoModelForCausalLM.from_pretrained(

    model_name,

    quantization_config=bnb_config,

    device_map="auto"
)



print("✅ meta-llama/Llama-3.2-1B loaded successfully")

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

✅ meta-llama/Llama-3.2-1B loaded successfully


In [22]:
# =====================================================
# STEP 10: Configure LoRA Adapter
# =====================================================


from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)


# Prepare quantized model for training
model = prepare_model_for_kbit_training(model)


# LoRA configuration
lora_config = LoraConfig(

    # Rank of LoRA matrices
    r=16,

    # Scaling factor
    lora_alpha=32,


    # Target attention layers
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],


    # Prevent overfitting
    lora_dropout=0.1,


    bias="none",


    # Because Llama is causal language model
    task_type="CAUSAL_LM"
)



# Attach LoRA adapters
model = get_peft_model(
    model,
    lora_config
)



print("✅ LoRA adapter added successfully")

print("--------------------------------")

model.print_trainable_parameters()

✅ LoRA adapter added successfully
--------------------------------
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


In [23]:
# =====================================================
# STEP 11: Configure Fine-Tuning Parameters
# =====================================================


from transformers import TrainingArguments


training_args = TrainingArguments(

    # Save checkpoints here
    output_dir="/content/drive/MyDrive/llama3.2-router-finetuning/checkpoints",


    # Number of times model sees dataset
    num_train_epochs=2,


    # Batch size for T4 GPU
    per_device_train_batch_size=2,


    # Simulates larger batch size
    gradient_accumulation_steps=4,


    # Learning speed
    learning_rate=2e-4,


    # Save model every 100 steps
    save_steps=100,


    # Show training logs
    logging_steps=25,


    # Optimizer for QLoRA
    optim="paged_adamw_8bit",


    # Mixed precision training
    fp16=True,


    # Remove unused columns
    remove_unused_columns=False,


    # Don't upload automatically
    report_to="none"
)



print("✅ Training arguments configured")

✅ Training arguments configured


In [27]:
# =====================================================
# STEP 12 (FIXED): Create Fine-Tuning Trainer
# =====================================================


from trl import SFTTrainer, SFTConfig


sft_config = SFTConfig(

    output_dir="/content/drive/MyDrive/llama3.2-router-finetuning/checkpoints",

    num_train_epochs=2,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=25,

    save_steps=100,

    max_length=512,


    # IMPORTANT FIX
    fp16=False,
    bf16=False,


    optim="paged_adamw_8bit",

    report_to="none"
)


trainer = SFTTrainer(

    model=model,

    train_dataset=train_dataset,

    args=sft_config
)


print("✅ Fixed trainer created successfully")

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Fixed trainer created successfully


In [28]:
# =====================================================
# STEP 13: Start Fine-Tuning
#
# Llama 3.2:1B will now learn:
# =====================================================


trainer.train()


print("🎉 Fine-tuning completed successfully")

Step,Training Loss
25,1.319300
50,0.291814
75,0.198737
100,0.167669
125,0.159901
150,0.145786
175,0.137219
200,0.137157
225,0.135504
250,0.117837


🎉 Fine-tuning completed successfully


In [29]:
# =====================================================
# STEP 14: Save Fine-Tuned Router Model
#
# Purpose:
# Save LoRA adapter + tokenizer
# permanently in Google Drive
#
# So we don't train again
# =====================================================


SAVE_PATH = "/content/drive/MyDrive/llama3.2-router-finetuning/final_model"


# Save LoRA fine tuned adapter
model.save_pretrained(SAVE_PATH)


# Save tokenizer
tokenizer.save_pretrained(SAVE_PATH)


print("✅ Fine tuned model saved successfully")

print("Saved Location:")
print(SAVE_PATH)

✅ Fine tuned model saved successfully
Saved Location:
/content/drive/MyDrive/llama3.2-router-finetuning/final_model


In [30]:
# =====================================================
# STEP 15: Test Fine-Tuned Router Model
#
# Purpose:
# Send a new user request and check
# router decision generated by model
# =====================================================


# Put model into evaluation mode
model.eval()


test_prompt = """
### Instruction:
You are an intelligent AI model router.

Analyze the user's request and decide:
1. Task category
2. Best model to handle the request
3. Backup model if the first model fails

Return only JSON output.

### User Request:
Create a Python REST API using FastAPI with authentication.

### Response:
"""


# Convert text into tokens
inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to("cuda")


# Generate response
outputs = model.generate(

    **inputs,

    max_new_tokens=100,

    temperature=0.2,

    do_sample=True
)


# Decode output
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)


print(response)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



### Instruction:
You are an intelligent AI model router.

Analyze the user's request and decide:
1. Task category
2. Best model to handle the request
3. Backup model if the first model fails

Return only JSON output.

### User Request:
Create a Python REST API using FastAPI with authentication.

### Response:
{"task_type": "coding", "primary_model": "mixtral", "fallback_model": "qwen"}
